## Load Dependencies

In [ ]:
from tensorflow import keras
import tensorflow as tf
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import cv2
import os
from tqdm import tqdm 
from sklearn.model_selection import train_test_split

## Load Data

In [ ]:
NUMBER_OF_PICTURES = 1818 # Total number of images in the dataset
SIZE=200 # Change image size if needed
DATASET = 'dataset' # Change this to the path of dataset image folder if needed
DATASET_EXCEL = 'dataset.xlsx' # Change this to the path of dataset excel file if needed

In [ ]:
def img_to_array(img):
    return np.asarray(img)

clean_data=[]
path = DATASET
files=os.listdir(path)
for i in tqdm(files):
    img=cv2.imread(path+'/'+i)  #Change 0 to 1 for color images
    # print(path+'/'+i)
    if(img.shape[0] > 0):
        img=cv2.resize(img,(SIZE, SIZE))
        img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        clean_data.append(img_to_array(img))
    else:
        print(img.shape)
        print(i)
        print(path+'/'+i)


In [ ]:
clean_data=clean_data[:NUMBER_OF_PICTURES]
plt.imshow(clean_data[0],cmap='binary')
new_clean=[]
for i in clean_data:
    i=i/255
    new_clean.append(i)

In [ ]:
for i in new_clean:
    x=i
    x[x<0.5]=0

In [ ]:
plt.imshow(new_clean[0],cmap='binary')

## Data analysis - Removing the noise

In [ ]:
arr_ind =  np.arange(NUMBER_OF_PICTURES)
mean_row=[]
for i in arr_ind:
    mean_row.append(np.mean(new_clean[i][120]))
mean_array=np.array(mean_row)

## Check if most of the information are inside this range 

In [ ]:
plt.scatter(arr_ind,mean_array)
plt.plot(arr_ind,mean_array)
plt.show()

## Cropping the images - only keeping the sky portion

In [ ]:
for i in range(len(new_clean)):
    new_clean[i]=new_clean[i][:120]

In [ ]:
df=pd.read_excel(DATASET_EXCEL, engine='openpyxl').head(NUMBER_OF_PICTURES)

In [ ]:
# print(df)
df.isnull().any()

In [ ]:
y=df['PM2.5']
y=np.asarray(y).astype('float32')
print(len(y))
print(type(y))
print(y)

## Data Distribution

In [ ]:
y = df['PM2.5']
y = np.array(y)


check_0_50 = 0 
check_50_100 = 0
check_100_150 = 0
check_150_200 = 0
check_200_250 = 0
check_250_300 = 0
check_300_more = 0  

j = 0
for i in y:
    j = j + 1
    if i < 50:
        check_0_50 = check_0_50 + 1
    elif i >= 50 and i<100:
        # print(j-1)
        check_50_100 = check_50_100 + 1
    elif i >= 100 and i<150:
        # print(j-1)
        check_100_150 = check_100_150 + 1
    elif i >= 150 and i<200:
        # print(j-1)
        check_150_200 = check_150_200 + 1
    elif i >= 200 and i<250:
        # print(j-1)
        check_250_300 = check_250_300 + 1
    elif i >= 250 and i<300:
        # print(j-1)
        check_50_100 = check_50_100 + 1
    elif i >= 300:
        # print(j-1)
        check_300_more = check_300_more + 1

arr = [check_0_50, check_50_100, check_100_150, check_150_200, check_200_250, check_250_300, check_300_more]
z = [0, 50, 100, 150, 200, 250, 300]

# plt.style.use('ggplot')
# plt.bar(x=np.arange(y.size),height=y)
plt.hist(y, color=(0.2, 0.4, 0.6, 0.6),  edgecolor='blue')
plt.xlabel('PM2.5 Index')
plt.ylabel('Number of Images')
plt.gcf().set_dpi(300)
# plt.grid()
plt.show()

In [ ]:
new_clean[0].shape

In [ ]:
X=new_clean
print(len(X))
X=np.asarray(X).astype('float32')

print(type(X))

In [ ]:
plt.imshow(new_clean[0],cmap='binary')

## Dataset Splitting

In [ ]:
TEST_SIZE = 0.10 # Change test size if needed
SEED = 42 # Change random seed if needed

x_train, x_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size = TEST_SIZE, random_state = SEED,shuffle=True)

In [ ]:
x_train = x_train.reshape(-1,120,200, 3)

x_test = x_test.reshape(-1,120,200, 3)

In [ ]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

## Model Development

In [ ]:
from tensorflow.keras.layers import Dense, Input, Dropout,Flatten, Conv2D
from tensorflow.keras.layers import BatchNormalization, Activation, MaxPooling2D, UpSampling2D, AveragePooling2D
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import plot_model

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.ZeroPadding2D(padding=(3, 3), input_shape=(120,200,3)),
    # First Conv2d Block
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    # Second Conv2d Block
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2,2), 
    # Third Conv2d Block
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2,2),
    # Fourth Conv2d Block
    tf.keras.layers.Conv2D(256, (3,3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2,2),
    # Fifth and Final Conv2d Block
    tf.keras.layers.Conv2D(512, (3,3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(2,2),
    # Fully Connected Layer
    tf.keras.layers.Flatten(), 
    tf.keras.layers.Dense(128, activation='relu'), 
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1,activation='linear')  
])


## Model Summary

In [ ]:
model.summary()

## Visualize the Model

In [ ]:
# from PIL import ImageFont
# from collections import defaultdict

# font = ImageFont.truetype("arial.ttf", 18) 

# color_map = defaultdict(dict)
# color_map[tf.keras.layers.Conv2D]['fill'] = 'orange'
# color_map[tf.keras.layers.Dropout]['fill'] = 'silver'
# color_map[tf.keras.layers.MaxPooling2D]['fill'] = 'red'
# color_map[tf.keras.layers.Dense]['fill'] = 'blue'
# color_map[tf.keras.layers.Flatten]['fill'] = 'purple'
# color_map[tf.keras.layers.BatchNormalization]['fill'] = 'lightgreen'

# import visualkeras
# visualkeras.layered_view(model, legend=True,font=font,color_map=color_map)#, draw_volume=False)

## Images with Unique Values

In [ ]:
y = df['PM2.5']
y = np.array(y)

unique, counts = np.unique(y, return_counts=True)

xpoints = np.array(unique)
ypoints = np.array(counts)

plt.hist(xpoints)
plt.xlabel('PM2.5 Index')
plt.ylabel('Frequency of Images')
plt.gcf().set_dpi(300)
plt.show()

## Model Compilation

In [ ]:
from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.optimizers import RMSprop

LR = 0.000001 # Change learning rate if needed

optimum=Adam(learning_rate=LR)
model.compile(optimizer=optimum, loss='mean_absolute_error',metrics=['mean_squared_error',tf.keras.metrics.RootMeanSquaredError()])

## Fitting The Model

In [ ]:
import time

BATCH_SIZE = 8 # Change batch size if needed
EPOCHS = 350 # Change number of epochs if needed
VALIDATION_SPLIT = 0.1 # Change validation split if needed

start = time.process_time()
hist = model.fit(x_train, y_train, 
           batch_size=BATCH_SIZE, epochs=EPOCHS, validation_split=VALIDATION_SPLIT)
print(time.process_time() - start)

## Print Predicted values

In [ ]:
predictions = model.predict([x_test])
pred=[]
for i in predictions:
    print(int(i[0]))
    pred.append(int(i[0]))

## Print True Values

In [ ]:
for i in y_test:
    print(i)

## Mean Difference from the true value

In [ ]:
diff=[]
for i in range(len(pred)):
    diff.append(abs(y_test[i]-pred[i]))
print(sum(diff) / len(diff))

## Print Differences

In [ ]:
for i in diff:
    print(i)

### Test Values

In [ ]:
eva = model.evaluate(x_test, y_test)
print(eva)

### MSE Train vs Validation Curve

In [ ]:
sns.set()

plt.plot(hist.history['mean_squared_error'], label='Train')
plt.plot(hist.history['val_mean_squared_error'], label='Validation')
plt.legend()
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


### MAE Train vs Validation Curve

In [ ]:
plt.plot(hist.history['loss'], label='Train')
plt.plot(hist.history['val_loss'], label='Validation')
plt.legend()
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


### RMSE Train vs Validation Curve

In [ ]:
plt.plot(hist.history['root_mean_squared_error'], label='Train')
plt.plot(hist.history['val_root_mean_squared_error'], label='Validation')
plt.legend()
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
print(len(x_test)) # Test data size

### Near Values

In [ ]:
check_20 = 0 #check how many differences are less or equal 20
check_30 = 0 #check how many differences are less or equal 30
check_40 = 0 #check how many differences are less or equal 40
check_50 = 0 #check how many differences are less or equal 60
check_10 = 0 #check how many differences are less 10
check_50_more = 0 #check how many differences are more than 50
check_0 = 0
check_below_50 = 0
total=0
# for i in diff:
#     if i == 0:
#         check_0 = check_0 + 1
#         total+=1
#     elif i <= 10:
#         check_10 = check_10 + 1
#         total+=1
#     elif i <= 20:
#         total+=1
#         check_20 = check_20 + 1
#     elif i <= 30:
#         total+=1
#         check_30 = check_30 + 1
#     elif i <= 40:
#         total+=1
#         check_40 = check_40 + 1
#     elif i <= 50:
#         total+=1
#         check_50 = check_50 + 1
#     elif i > 50:
#         check_50_more = check_50_more + 1
#         # print("Too far")
j = 0
for i in diff:
    j = j + 1
    if i < 50:
        total+=1
        check_below_50 = check_below_50 + 1
    elif i >= 50:
        # print(j-1)
        check_50_more = check_50_more + 1
        # print("Too far")


# print(check_0)
# print(check_10)
# print(check_20)
print("# of pictures of AQI Difference of less than 50: ", check_below_50)
print("# of pictures of AQI Difference of 50 or more: ", check_50_more)
# print('Near:',total)

In [ ]:
near_percentage = (total/len(x_test) )* 100 # This percent of images can be considered as truly close to actual value since 50 difference is.

In [ ]:
print(near_percentage)

### Test Dataset Amount

In [ ]:
len(x_test)

In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])

In [ ]:
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
my_rho = np.corrcoef(predic, obser)

print(my_rho)

In [ ]:
# losas= tf.keras.metrics.mean_absolute_error(obser, predic) # obsolete in tf 2.18
mae = tf.keras.metrics.MeanAbsoluteError()
mae.update_state(obser, predic)          # obser and predic must have identical shape
losas = mae.result().numpy()             # scalar MAE

In [ ]:
losas

## $R^2$ Score

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(obser, predic)

In [ ]:
print(r2)

In [ ]:
print(x_test)

## Test Images with observed and predicted values

In [ ]:
from PIL import Image as im

val = 0

data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 3
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 4
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 5
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 6
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 7
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 10
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 11
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 12
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 14
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 15
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 16
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 17
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 18
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 19
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 20
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 21
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 72
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 172
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
from PIL import Image as im

val = 181
  
data = im.fromarray((x_test[val]* 255).astype(np.uint8))
plt.imshow(data,cmap='binary')
actual_val = 'Observed Value:  ' + str(pred[val])
pred_val = 'Predicted Value: ' + str(y_test[val])
plt.xlabel(actual_val)
plt.ylabel(pred_val)
plt.gcf().set_dpi(300)
# print(x_test[0][0])

In [ ]:
# model.save("aqi_model.h5")   # HDF5 format
# model.save("aqi_model.keras")   # Recommended modern format